# Module 1 — REST API: guided beginner activities

In this activity, you will explore the supplied Flask URL-shortener service by sending HTTP requests and examining its responses. The service stores short-ID-to-URL mappings in memory.

Work in order: **read → predict → run → inspect → explain**. This notebook sends requests only to your participant-local service at `127.0.0.1:5000`; do not change service or infrastructure settings.

The beginner track supplies the complete application in `app.py`. You will not need to build another server or add helper functions. The notebook's code cells are your HTTP client.

## Outcomes derived from Assignment 1

The repository contains no separate formal learning-objective list. These activities map to the assignment's stated expectations; they are not a replacement set of official objectives. By the end, you should be able to:

1. Explain how an HTTP method and path select an operation on a URL mapping.
2. Send requests, read JSON bodies and interpret status codes.
3. Explain how Flask routes, request parsing, the in-memory dictionary and response creation fit together.
4. Describe the beginner app's URL-checking and ID-generation helpers, including their limitations.
5. Test the service and explain how you would extend the design so users manage their own mappings.

| Assignment expectation | Practised in | Evidence/checkpoint |
|---|---|---|
| Interpret and demonstrate the endpoint specification | Activities 1–5 | Correct request method/path, status and observed state change |
| Explain input checking and ID design | Activities 6–7 | Explain what each helper accepts/returns and what it does not guarantee |
| Discuss design choices and multi-user extension | Review | Explain trade-offs without adding identity or account functionality |


## 1. The API as a set of operations

An API is an interface that another program can call. Here, the resource is a stored URL mapping: a short ID points to a longer URL. The HTTP method describes what the client asks the service to do.

| Method and path | Meaning in this workshop | Expected result for the supplied beginner app |
|---|---|---|
| `GET /` | List stored IDs | `200`, JSON containing `keys` |
| `POST /` | Create a mapping; JSON body uses `value` | `201`, JSON containing a new `id` |
| `GET /<id>` | Look up one mapping | `301` with the URL in JSON, or `404` if unknown |
| `PUT /<id>` | Replace an existing mapping; JSON body uses `url` | `200`, or `400`/`404` on the errors described in the code |
| `DELETE /<id>` | Delete one mapping | `204` with an empty body, or `404` if unknown |
| `DELETE /` | Clear every mapping | `404` in this assignment, even though all mappings are cleared |

The last two `404`/`301` conventions are assignment-specific. A status code is part of the observable response; do not assume that it always has the most conventional meaning in a production API.


## 2. Start the supplied service

Open a **JupyterHub terminal** for this participant server. Start the provided app in that terminal and leave it running while you use the notebook.

```bash
cd /home/jovyan/Cloud-Computing-SOA/Module-1/beginner
python app.py
```

The service listens at `http://127.0.0.1:5000`. Your notebook and terminal must belong to the same participant JupyterHub server for this local address to work. If the terminal reports an error, stop and ask the organiser rather than changing deployment settings. Restarting the app clears its in-memory mappings.

**Predict:** immediately after a fresh start, what should `GET /` list?


### How the notebook will send requests

The `requests` package sends HTTP requests from Python. For example, `requests.get(url)` sends a GET request, and `requests.post(url, json=...)` sends a POST with a JSON body. Each call returns a **Response** object. Its `status_code` is the HTTP status, `headers` contains response headers, `text` is the body as text, and `json()` parses a JSON body into Python values. A `204` response has no body, so do not call `json()` on it.

The `timeout` argument limits how long the client waits for the local service. `allow_redirects=False` (used for lookup below) asks `requests` not to follow a `301` to the stored URL. This keeps the exercise focused on the local API response.


In [ ]:
import requests

BASE_URL = "http://127.0.0.1:5000"
print("API base address:", BASE_URL)


## 3. Read the collection: `GET /`

The route `GET /` asks for the collection—the set of mappings, not one specific URL. Run the request. Inspect the status and JSON body.

**Checkpoint:** after a fresh app start, expect status `200` and an empty `keys` list. If keys are already present, the server has not just been restarted; record what you observe rather than assuming the service is broken.


In [ ]:
response = requests.get(BASE_URL + "/", timeout=5)
print("Status:", response.status_code)
print("JSON body:", response.json())


### Connect the response to the Flask code

In `beginner/app.py`, the corresponding route is:

```python
@app.route("/", methods=["GET"])
def read_root():
    return jsonify({"keys": list(shared_dict.keys())}), 200
```

`@app.route` registers which method and path should call `read_root`. `shared_dict` is a Python dictionary holding `short_id -> URL`. `list(shared_dict.keys())` makes a list of its IDs. Flask's `jsonify(...)` converts the Python value into a JSON HTTP response; the `200` is the response status.

This is a code-reading example, not a new function to type into the notebook. **Check:** point to the route, the stored data, and the returned status in the snippet.


## 4. Create one mapping: `POST /`

The supplied app reads a JSON object with a `value` field. The value should be a URL. In `requests.post`, `json=...` supplies the JSON request body; it is not the URL itself. A successful creation returns a short ID.

**Predict:** what status should indicate that a resource was created? What information will you need to look it up next?


In [ ]:
target_url = "https://example.org/rest-api-practice"
created = requests.post(
    BASE_URL + "/",
    json={"value": target_url},
    timeout=5,
)
print("Status:", created.status_code)
print("JSON body:", created.json())


**Checkpoint:** expect `201` and a JSON object with an `id`. Do not assume the ID is a particular value: it depends on the app's ID-generation function. Save the returned value for the next request.


In [ ]:
short_id = created.json()["id"]
print("ID returned by POST:", short_id)


## 5. Look up the new mapping: `GET /<id>`

In a Flask route such as `/<id>`, the portion in angle brackets is a variable route segment. Flask passes the requested value to the handler as the `id` argument. The lookup handler reads the dictionary and returns either the saved URL or a not-found response.

We set `allow_redirects=False` so the client shows the API's `301` response rather than following it. This is especially important for understanding a redirect: the target is the URL value, not another endpoint of this local service.


In [ ]:
lookup = requests.get(
    BASE_URL + "/" + short_id,
    allow_redirects=False,
    timeout=5,
)
print("Status:", lookup.status_code)
print("JSON body:", lookup.json())
print("Location header:", lookup.headers.get("Location"))


**Checkpoint:** the supplied beginner app returns `301` and a JSON body with `value` equal to the URL you posted. It does not set a `Location` header. A different implementation may use a `Location` header; do not follow it during this activity.

Try a made-up ID and explain the difference between a known and unknown resource.


In [ ]:
missing_id = short_id + "-not-found"
missing = requests.get(
    BASE_URL + "/" + missing_id,
    allow_redirects=False,
    timeout=5,
)
print("Status:", missing.status_code)
print("JSON body:", missing.json())


## 6. Update the URL: `PUT /<id>`

For an update, the app expects a JSON object with the field `url` (unlike POST, which uses `value`). `request.get_json(...)` in the Flask handler reads the request body as Python data. The handler checks whether the ID exists, checks the new value, and replaces the dictionary value.

**Predict:** what should a later GET return after a successful update?


In [ ]:
updated_url = "https://example.org/updated-practice"
updated = requests.put(
    BASE_URL + "/" + short_id,
    json={"url": updated_url},
    timeout=5,
)
print("Status:", updated.status_code)
print("JSON body:", updated.json())


In [ ]:
lookup_after_update = requests.get(
    BASE_URL + "/" + short_id,
    allow_redirects=False,
    timeout=5,
)
print("Status:", lookup_after_update.status_code)
print("JSON body:", lookup_after_update.json())


### Test the PUT error cases

The following requests target the existing ID with invalid URL text and a missing ID with valid URL text. Predict `400` and `404`, then inspect both status codes and bodies. Check that the invalid update did not alter the stored mapping.


In [ ]:
invalid_update = requests.put(
 BASE_URL + "/" + short_id,
 json={"url": "not a URL"},
 timeout=5,
)
print("Invalid URL update status:", invalid_update.status_code)
print("Body:", invalid_update.json())

missing_update = requests.put(
 BASE_URL + "/" + missing_id,
 json={"url": updated_url},
 timeout=5,
)
print("Missing ID update status:", missing_update.status_code)
print("Body:", missing_update.json())

verify_after_errors = requests.get(
 BASE_URL + "/" + short_id,
 allow_redirects=False,
 timeout=5,
)
print("Existing mapping after rejected update:", verify_after_errors.json())


**Checkpoint:** the update should return `200`, and the subsequent lookup should contain `updated_url`. `jsonify` creates JSON responses for these handlers. Check both status and content: one alone does not show that the stored value changed.


## 7. Understand the beginner URL-checking helper

The app defines `is_it_an_url(value)` as a helper: it takes one input value and returns a Python Boolean, `True` or `False`. The supplied implementation is:

```python
def is_it_an_url(value):
    return isinstance(value, str) and (
        value.startswith("http://") or value.startswith("https://")
    )
```

`isinstance(value, str)` checks that the input is text. `startswith(...)` checks only the beginning of that text. Thus this helper rejects some bad input, but it does **not** establish that the rest of the text is a valid URL or even contains a host. We will test both a clearly invalid string and the edge case `http://`.


In [ ]:
def is_it_an_url(value):
    """Workshop helper: True when text starts with http:// or https://."""
    return isinstance(value, str) and (
        value.startswith("http://") or value.startswith("https://")
    )

for example in ["https://example.org", "not a URL", "http://"]:
    print(repr(example), "->", is_it_an_url(example))


The printed results follow directly from the prefix check: `http://` passes even though it has no host. The same helper is used by POST and PUT. Next, observe its effect through the HTTP API. `response.json()` is appropriate only when the response contains JSON; the beginner app's error and creation responses do.


In [ ]:
clearly_invalid = requests.post(
    BASE_URL + "/", json={"value": "not a URL"}, timeout=5
)
print("Clearly invalid — status:", clearly_invalid.status_code)
print("Body:", clearly_invalid.json())

prefix_only = requests.post(
    BASE_URL + "/", json={"value": "http://"}, timeout=5
)
print("Prefix but no host — status:", prefix_only.status_code)
print("Body:", prefix_only.json())


**Checkpoint:** `not a URL` should receive `400`; `http://` is accepted by this beginner helper and receives `201`. This exposes a limitation in the supplied beginner app, not a URL worth using. The second request creates a disposable test mapping; we will clear it near the end.

Assignment 1 requires a **regular-expression URL check** for full credit. The current prefix check is not that check. `re.fullmatch(pattern, value)` is a Python operation that succeeds only when the entire string matches a pattern. The short example below demonstrates the idea; it is **not** a complete URL validator or a full-credit solution.


In [ ]:
import re

scheme_example = re.compile(r"https?://\S+")
for example in ["https://example.org", "not a URL", "http://"]:
    print(repr(example), "->", scheme_example.fullmatch(example) is not None)


In this example, `https?` means `http` followed by an optional `s`; `://` is literal text; `\S+` means one or more non-whitespace characters. `fullmatch` checks the whole input. This still does not verify that a host is well formed: for example, it can accept text that begins with the scheme but is not a real domain.

**Supported practice:** using the assignment tests as examples, write down at least three valid and three invalid inputs your validator should distinguish. Explain why a prefix-only check or this teaching pattern is insufficient. Do not describe either simple check as production-grade validation.


## 8. Understand the beginner ID-generation helper

The supplied `generate_short_id(url)` takes a URL string and returns the first six hexadecimal characters of an MD5 digest of that text. The relevant implementation is:

```python
def generate_short_id(url):
    return hashlib.md5(url.encode()).hexdigest()[:6]
```

`encode()` converts the text into bytes for `hashlib`; `hexdigest()` produces hexadecimal text; `[:6]` keeps six characters. The same input always gives the same result. A six-character digest cannot guarantee that distinct URLs receive distinct IDs. Compare this limitation with the assignment request for unique, as-short-as-possible IDs and its tests for conflicts after deletion.


In [ ]:
import hashlib

def beginner_id_example(url):
    """Mirror the app's six-character hash formula for learning."""
    return hashlib.md5(url.encode()).hexdigest()[:6]

sample_url = "https://example.org/rest-api-practice"
print("URL:", sample_url)
print("Example ID:", beginner_id_example(sample_url))
print("Same input, same ID:", beginner_id_example(sample_url) == beginner_id_example(sample_url))


This cell mirrors the app formula but does not prove collision-free IDs. Distinct URLs can share the same six-character prefix, and a repeated URL deterministically receives the same ID again. Assignment 1 asks for unique, as-short-as-possible identifiers and tests conflicts after deletion. Discuss what guarantees your design would need; a small passing sample is not proof of uniqueness.

**Checkpoint:** in one or two sentences, distinguish what the beginner helper returns from the uniqueness property the assignment asks you to explain.


## 9. Delete one mapping and handle missing resources

`DELETE /<id>` asks the service to remove one entry. The supplied app returns `204` on success, with an empty response body. For an empty-body response, inspect `status_code` and `text`; do not try to parse JSON.


In [ ]:
deleted = requests.delete(BASE_URL + "/" + short_id, timeout=5)
print("Status:", deleted.status_code)
print("Body text:", repr(deleted.text))


### Try deleting the same ID again

After the successful deletion, repeat the request for that ID. Predict the status and inspect the not-found body.


In [ ]:
delete_missing = requests.delete(BASE_URL + "/" + short_id, timeout=5)
print("Repeated DELETE status:", delete_missing.status_code)
print("Body:", delete_missing.json())


In [ ]:
get_after_delete = requests.get(
    BASE_URL + "/" + short_id,
    allow_redirects=False,
    timeout=5,
)
print("GET after delete — status:", get_after_delete.status_code)
print("Body:", get_after_delete.json())


**Checkpoint:** deletion returns `204`; the following lookup returns `404`. This is evidence for the two requests you just made. It does not mean that every `404` response has the same meaning—inspect the operation and body too.


## 10. Clear the collection, then run the supplied tests

`DELETE /` clears **all** mappings in this participant-local app. This assignment specifies a `404` response for that operation even though the entries are removed. The next GET lets you check the side effect. Do not run this request against any shared service.


In [ ]:
cleared = requests.delete(BASE_URL + "/", timeout=5)
print("DELETE / status:", cleared.status_code)
print("DELETE / body:", cleared.json())

after_clear = requests.get(BASE_URL + "/", timeout=5)
print("GET / status:", after_clear.status_code)
print("GET / body:", after_clear.json())


**Checkpoint:** expect `DELETE /` to return `404`, followed by `GET /` returning `200` with an empty `keys` list. The status is unusual, but matches the assignment contract.

The repository also supplies `tests/test_api.py`. You can run it from the tests directory in a JupyterHub terminal after the guided activity:

```bash
cd /home/jovyan/Cloud-Computing-SOA/Module-1/tests
python test_api.py
```

The script sends requests to the running app, tests API behaviour, and ends each test by calling `DELETE /`. It changes the app's mappings and clears them; use it only when you are ready to discard this activity's data. Tests cover selected cases, not every possible input or production readiness. Read failures as clues about which observable behaviour differed.


## 11. Supported practice and review

Try one more mapping using a different URL. Before each request, write down the method, path, JSON body (if any), and status/body you expect. Then use the same direct `requests` calls above to create it, list IDs, look it up, update it and delete it. Do not hard-code an ID; use the value returned by POST.

### Reflection

1. Trace the successful POST: what did the notebook send, which Flask route handled it, what did the app store, and what did the response contain?
2. Why does PUT use the body field `url` while POST uses `value` in this supplied app?
3. What does `is_it_an_url` check, and what does the `http://` observation show it misses?
4. Why does a six-character hash not prove unique IDs? What guarantee would an alternative ID-generation strategy need to provide?
5. Which response uses `301`, `204`, `400` and `404` in this assignment? Which result seems unusual, and why?
6. Conceptually, what would need to change for multiple users to create and remove only their own mappings? Consider ownership information and checking ownership on update/delete; this is a design reflection, not a request to add user accounts or authentication.

### Finish checklist

- [ ] I can match each method and path to its action.
- [ ] I inspected an ID returned by POST instead of guessing one.
- [ ] I checked status and body, including a `204` with no body and a `404` case.
- [ ] I can explain the inputs, outputs and limits of the two beginner helpers.
- [ ] I understand that this app stores data in memory and the tests clear it.
- [ ] I can distinguish the supplied beginner implementation from the assignment's regex-validation and ID-uniqueness expectations.
